# DataMate — train 3000 / public test 1000

Một notebook, cùng root `predict.sh` cho cả hai split. Chọn GPU rồi **Run all**.
Code lấy từ GitHub; weights/runtime được tải ở setup từ revision/hash đã pin.
Không cần runtime ZIP. Dữ liệu đã có trên Drive, notebook không sửa input gốc:

```text
MyDrive/DataMate/
├── train.zip          # auto ưu tiên ZIP nếu có
├── train/             # dùng nếu không có ZIP, hoặc INPUT_SOURCE="folder"
│   ├── questions.jsonl, answers.jsonl, rulebook.md, calendar.csv
│   └── data/
└── public_test/
    ├── questions.jsonl, rulebook.md, calendar.csv
    └── data/
```

- `SPLIT="train"`: chạy đủ3000 trước, sau đó mới chấm cả3000 labels nếu EVALUATE_TRAIN=True.
- `SPLIT="public_test"`: chạy đủ1000, không đọc labels hoặc chạy evaluator; tải JSONL để tự nộp web.
- `CHUNK_SIZE=100`: checkpoint sau mỗi100 câu. Cùng RUN_ID/source/config để resume;
  đổi RUN_ID khi chạy thử độc lập. Train/public có namespace riêng, không trộn checkpoints.
- Data preparation, cache và model ở SSD `/content`; Drive giữ dataset và checkpoint/results.
  Phần đang chạy dở chưa checkpoint có thể phải chạy lại sau khi mất VM.
- Repo private: thêm Colab Secret `GITHUB_TOKEN` (Contents:read) hoặc nhập khi được hỏi.
  Ref GitHub phải đã có helper train/public mới; không tự push thay đổi local.

Notebook chỉ chạy inference, không training/tuning model. Toàn train được chấm theo yêu cầu;
không gọi đây là blind holdout. Colab chưa được nghiệm thu hosted và không chứng nhận
Docker/Blackwell/SLA. Không tự upload hoặc submit đến BTC.

In [ ]:
import ast
import getpass
import hashlib
import json
import os
import shutil
import signal
import subprocess
import sys
import tempfile
from pathlib import Path

from google.colab import drive, files, userdata

SPLIT = "train"  # "train" (3000) hoặc "public_test" (1000).
DATA_ROOT = Path("/content/drive/MyDrive/DataMate")
INPUT_SOURCE = "auto"  # "auto": ZIP nếu có, còn lại folder; hoặc "zip" / "folder".
RUN_ID = "baseline-v2"  # Giữ cho resume; đổi khi chạy độc lập.
CHUNK_SIZE = 100  # Chọn 3000/1000 nếu muốn một process cho cả split.
EVALUATE_TRAIN = True
DOWNLOAD_PUBLIC = True
RUN_QUALITY_CHECKS = True
REPO_URL = "https://github.com/TruongQuangPhat/Zalo-AI-Challenge.git"
REPO_REF = "main"  # Dùng commit SHA đã in ở lần đầu để resume nếu main đã thay đổi.
PROJECT = Path("/content/Zalo-AI-Challenge")
COUNTS = {"train": 3000, "public_test": 1000}
assert SPLIT in COUNTS and 1 <= CHUNK_SIZE <= COUNTS[SPLIT]
assert RUN_ID and all(c.isalnum() or c in "-_." for c in RUN_ID) and RUN_ID not in {".", ".."}
drive.mount("/content/drive")
assert DATA_ROOT.is_dir(), f"Không thấy {DATA_ROOT}; chỉnh DATA_ROOT theo folder DataMate của bạn."
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
# Source setup only. No token in URL, Git config, notebook output or persistent files.
if not PROJECT.exists():
    clone_env = os.environ.copy()
    clone_env["GIT_TERMINAL_PROMPT"] = "0"
    public = subprocess.run(
        ["git", "-c", "credential.helper=", "ls-remote", REPO_URL, "HEAD"],
        env=clone_env,
        capture_output=True,
    )
    if public.returncode == 0:
        subprocess.run(
            ["git", "-c", "credential.helper=", "clone", REPO_URL, str(PROJECT)],
            env=clone_env,
            check=True,
        )
    else:
        try:
            token = userdata.get("GITHUB_TOKEN")
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            token = getpass.getpass("GitHub token Contents:read cho repo private: ")
        assert token, "Cần token read-only để clone repo private."
        clone_env["DATAMATE_GITHUB_TOKEN"] = token
        try:
            with tempfile.TemporaryDirectory() as temporary:
                askpass = Path(temporary) / "askpass"
                askpass.write_text(
                    "#!/usr/bin/env python3\nimport os,sys\n"
                    'print("x-access-token" if "username" in sys.argv[1].lower() '
                    'else os.environ["DATAMATE_GITHUB_TOKEN"])\n'
                )
                askpass.chmod(0o700)
                clone_env["GIT_ASKPASS"] = str(askpass)
                subprocess.run(
                    ["git", "-c", "credential.helper=", "clone", REPO_URL, str(PROJECT)],
                    env=clone_env,
                    check=True,
                )
        finally:
            clone_env.pop("DATAMATE_GITHUB_TOKEN", None)
            del token
    subprocess.run(["git", "checkout", "--detach", REPO_REF], cwd=PROJECT, check=True)
    del clone_env
else:
    assert (
        subprocess.check_output(
            ["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True
        ).strip()
        == REPO_URL
    )
    subprocess.run(
        [
            "git",
            "diff",
            "--exit-code",
            "HEAD",
            "--",
            "src",
            "scripts",
            "predict.sh",
            "pyproject.toml",
            "uv.lock",
        ],
        cwd=PROJECT,
        check=True,
    )
SOURCE_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True
).strip()
expected_commit = subprocess.check_output(
    ["git", "rev-parse", REPO_REF + "^{commit}"], cwd=PROJECT, text=True
).strip()
assert SOURCE_COMMIT == expected_commit, "Ref đã đổi; dùng runtime mới hoặc ref của checkpoint cũ."
helper = PROJECT / "scripts/colab_pipeline.py"
definitions = {
    node.name: node
    for node in ast.parse(helper.read_text()).body
    if isinstance(node, ast.FunctionDef)
}
assert "stage_from_root" in definitions and any(
    arg.arg == "split_name" for arg in definitions["run_pipeline"].args.kwonlyargs
), (
    "GitHub ref chưa có helper train-public mới. Publish thay đổi vào ref trước. "
    "Không dùng helper train3000 cũ cho public."
)
print("Source commit:", SOURCE_COMMIT)

In [ ]:
# The Colab kernel can use another Python; every project command uses isolated Python3.11.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.12.0"], check=True)
subprocess.run(["uv", "python", "install", "3.11"], check=True)
subprocess.run(
    [
        "uv",
        "sync",
        "--locked",
        "--python",
        "3.11",
        "--no-default-groups",
        "--group",
        "dev",
        "--group",
        "llm",
        "--no-install-package",
        "llama-cpp-python",
    ],
    cwd=PROJECT,
    check=True,
)
PYTHON = PROJECT / ".venv/bin/python"
ARTIFACTS = PROJECT / ".artifacts"
ARTIFACTS.mkdir(exist_ok=True)
MODEL_DIR = ARTIFACTS / "models/qwen2.5-1.5b-instruct"
MODEL = MODEL_DIR / "model.gguf"

# Stream GGUF + upstream prebuilt CUDAwheel, verify SHA before use, install locked CUDA libs.
subprocess.run(
    [str(PYTHON), "-m", "scripts.colab_pipeline", "setup", "--directory", str(MODEL_DIR)],
    cwd=PROJECT,
    check=True,
)

In [ ]:
# Optional CPU/synthetic quality gates; never native answer feedback.
quality_log = ARTIFACTS / "colab-quality.txt"
if RUN_QUALITY_CHECKS:
    quality_commands = [
        [str(PYTHON), "-m", "ruff", "check", "src", "scripts", "tests"],
        [str(PYTHON), "-m", "ruff", "format", "--check", "src", "scripts", "tests"],
        ["uv", "lock", "--check", "--offline"],
        [str(PYTHON), "-m", "pytest", "-q", "--basetemp", str(ARTIFACTS / "pytest-colab")],
    ]
    with quality_log.open("w") as log:
        for command in quality_commands:
            result = subprocess.run(command, cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT)
            if result.returncode:
                print(quality_log.read_text()[-8000:])
                raise RuntimeError("Quality gate failed; stop before inference.")
    print(quality_log.read_text()[-2000:])
    shutil.rmtree(ARTIFACTS / "pytest-colab", ignore_errors=True)
else:
    print("Quality gates explicitly skipped; inference validation remains mandatory.")

In [ ]:
# Reuse the source-input allowlist; no labels enter the inference directory.
staging = json.loads(
    subprocess.check_output(
        [
            str(PYTHON),
            "-m",
            "scripts.colab_pipeline",
            "inputs",
            "--root",
            str(DATA_ROOT),
            "--workspace",
            str(ARTIFACTS / "colab-inputs"),
            "--split",
            SPLIT,
            "--source",
            INPUT_SOURCE,
        ],
        cwd=PROJECT,
        text=True,
    )
)
DATA = Path(staging["data_dir"])
TRAIN_ANSWERS = Path(staging["raw_dir"]) / "answers.jsonl" if SPLIT == "train" else None
if SPLIT == "train" and EVALUATE_TRAIN:
    assert TRAIN_ANSWERS.is_file(), (
        "Train source thiếu answers.jsonl; chọn INPUT_SOURCE='folder' nếu folder train có labels."
    )
assert staging["questions"] == COUNTS[SPLIT] and not (DATA / "answers.jsonl").exists()
RUN = ARTIFACTS / "colab-runs" / SPLIT / RUN_ID
CACHE = ARTIFACTS / "colab-cache" / SPLIT
CHECKPOINTS = DATA_ROOT / "results" / SPLIT / RUN_ID
print("Split:", SPLIT, "| questions:", staging["questions"], "| source:", staging["source_kind"])
print("Inference input on SSD:", DATA)
print("Checkpoint/results:", CHECKPOINTS)

Cell tiếp chạy đủ mọi ID của split qua cùng `predict.sh`, lưu checkpoint mỗi CHUNK_SIZE.
Đừng đổi source/input/model/chunk size khi resume cùng RUN_ID. Cache/business facts ở SSD
và được kiểm provenance/checksum; Drive chỉ nhận results/checkpoint, không đọc SQLite từ Drive.
Public không được phân bucket train hoặc chấm nhãn. Source ref có cập nhật thì chọn lại
commit đã in ở checkpoint, hoặc RUN_ID mới để chạy một lượt độc lập.

In [ ]:
def run_command(command):
    process = subprocess.Popen(command, cwd=PROJECT, start_new_session=True)
    try:
        if process.wait() != 0:
            raise RuntimeError("Command failed; giữ logs, không tải/nộp output chưa được kiểm tra.")
    finally:
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()


run_command(
    [
        str(PYTHON),
        "-m",
        "scripts.colab_pipeline",
        "run",
        "--data-dir",
        str(DATA),
        "--output-dir",
        str(RUN),
        "--cache-dir",
        str(CACHE),
        "--model",
        str(MODEL),
        "--checkpoint-dir",
        str(CHECKPOINTS),
        "--chunk-size",
        str(CHUNK_SIZE),
        "--split",
        SPLIT,
    ]
)
verification = json.loads((RUN / "verification.json").read_text())
assert verification["complete"] and verification["split"] == SPLIT
assert verification["questions"] == COUNTS[SPLIT]
print("Completed:", COUNTS[SPLIT], "questions | CUDA model loaded/offloaded and outputs verified.")
print(
    "Process wall seconds including model loading/preparation:",
    verification["process_seconds_including_loading_preparation"],
)

In [ ]:
# Only train enters this branch, after all3000 predictions have been verified.
if SPLIT == "train" and EVALUATE_TRAIN:
    REPORT = PROJECT / "reports/stage_results.json"
    EVALUATION_ID = "colab_train_" + RUN_ID
    saved_report = CHECKPOINTS / "stage_results.json"
    saved_proof = CHECKPOINTS / "evaluation-verification.json"
    if saved_report.exists() and saved_proof.exists():
        proof = json.loads(saved_proof.read_text())
        with saved_report.open("rb") as stream:
            assert hashlib.file_digest(stream, "sha256").hexdigest() == proof["report_sha256"]
        with TRAIN_ANSWERS.open("rb") as stream:
            assert hashlib.file_digest(stream, "sha256").hexdigest() == proof["answers_sha256"]
        assert proof["evaluation_id"] == EVALUATION_ID and proof["scored"] == 3000
        assert proof["submission_sha256"] == verification["submission_sha256"]
        assert proof["selection_sha256"] == verification["selection_sha256"]
        report_data = json.loads(saved_report.read_text())
        print(
            "Reused verified train evaluation:",
            report_data["stages"]["S6"]["summaries"][EVALUATION_ID],
        )
    else:
        assert not saved_report.exists() and not saved_proof.exists(), (
            "Evaluation copy chưa hoàn tất; giữ files và kiểm tra trước khi chạy lại."
        )
        run_command(
            [
                str(PYTHON),
                "-m",
                "scripts.colab_pipeline",
                "evaluate",
                "--answers",
                str(TRAIN_ANSWERS),
                "--run-dir",
                str(RUN),
                "--report",
                str(REPORT),
                "--evaluation-id",
                EVALUATION_ID,
            ]
        )
        shutil.copyfile(REPORT, saved_report)
        shutil.copyfile(RUN / "evaluation-verification.json", saved_proof)
else:
    print(
        "Public test: không đọc labels, không gọi evaluator."
        if SPLIT == "public_test"
        else "Train evaluation explicitly disabled."
    )

In [ ]:
for source, name in [
    (RUN / "selection.json", "selection.json"),
    (MODEL_DIR / "setup.json", "setup.json"),
    (ARTIFACTS / "colab-inputs" / (SPLIT + "-staging.json"), "staging.json"),
]:
    shutil.copyfile(source, CHECKPOINTS / name)
if RUN_QUALITY_CHECKS:
    shutil.copyfile(quality_log, CHECKPOINTS / "quality.txt")
print("Saved results:", CHECKPOINTS)
if SPLIT == "public_test":
    print(
        "Đây là public output1000 câu; tự nộp file submission.jsonl ở portal DataMate → Submission."
    )
    if DOWNLOAD_PUBLIC:
        files.download(str(CHECKPOINTS / "submission.jsonl"))
else:
    print("Train output3000 câu để đánh giá local, không nộp file này cho public leaderboard.")

Train score dùng evaluator local, chưa chứng nhận tương đương scorer BTC. Báo đủ mẫu số
3000: correct/wrong/abstain, PA và coverage; không gọi 1224 câu từng giữ lại là blind holdout
sau khi đã chấm. Public chỉ kiểm completeness/types/logs, không suy accuracy từ format pass.

Không gọi API inference; setup downloads hoàn tất trước predict. Cùng fixed GGUF, seed0,
temperature0, runtime version. Mỗi chunk reload model và thời gian có tính loading/preparation;
không coi đây là single-process SLA. Colab GPU/session có thể bị thu hồi; checkpoint chỉ cứu
phần đã xong, không né giới hạn dịch vụ. [Colab FAQ](https://research.google.com/colaboratory/faq.html).
Notebook chưa được chạy hosted bởi agent; kiểm output thực tế của từng gate.